# Case study: segments, maps and implausible listings among Berlin's Airbnb offers

*Session 11, blocks 2 and 3 practice. Author: course team, licence CC-BY-4.0. Data: Inside Airbnb, Berlin, snapshot of 26 June 2026 (CC BY 4.0).*

**Questions.** What kinds of Airbnb offers exist in Berlin? Where are they concentrated? What do the listings offer, and which listings are implausible? A city office that enforces the rules on short-term rentals, a tourism board or a new host would ask these questions; none of them comes with a target variable.

- **Part A (block 2):** k-means, hierarchical clustering and DBSCAN on six numbers per listing; evaluate with silhouette, stability and profiles; describe the clusters by district and room type, which the clustering did not see. DBSCAN on the map finds hot spots of listings.
- **Part B (block 3):** PCA of the amenity lists and a t-SNE map of the listings.
- **Part C (block 3):** implausible listings with Isolation Forest, LOF and a price ratio; do location clusters improve a gradient-boosting price model (Session 10)?

Theory: [page 2](../theory/02-hierarchical-dbscan-and-evaluation.md), [page 3](../theory/03-dimensionality-reduction.md), [page 4](../theory/04-anomaly-detection-and-unsupervised-features.md). Prepare the data once with `uv run python case-study/prepare_airbnb.py`.

*Ethics:* the data are scraped from public listing pages. Report in aggregate; do not name hosts, and do not publish lists of single listings outside the course.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.cluster import DBSCAN, KMeans
from sklearn.compose import make_column_transformer
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingRegressor, IsolationForest
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict, cross_val_score
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OrdinalEncoder, StandardScaler

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 14)
# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
DATA = ROOT / "case-study" / "data"

listings = pd.read_parquet(DATA / "airbnb" / "listings.parquet")
lat0, lon0 = 52.5219, 13.4132                                   # Alexanderplatz
listings["north_km"] = (listings["latitude"] - lat0) * 111.2
listings["east_km"] = (listings["longitude"] - lon0) * 111.2 * np.cos(np.radians(lat0))
listings["km_centre"] = np.hypot(listings["north_km"], listings["east_km"])
print(len(listings), "listings;", round(listings["price"].isna().mean(), 2), "without a price")

# Part A: what kinds of offers exist?

## A.1 Six numbers per listing

In [ ]:
d = listings.dropna(subset=["price", "minimum_nights"]).reset_index(drop=True)
features = pd.DataFrame({
    "log_price": np.log(d["price"]), "guests": d["accommodates"],
    "log_min_nights": np.log(d["minimum_nights"]), "availability_365": d["availability_365"],
    "log_reviews_ltm": np.log1p(d["number_of_reviews_ltm"]), "km_centre": d["km_centre"],
})
Z = StandardScaler().fit_transform(features)
features.describe().round(2)

**Question.** Why are price, minimum nights and reviews log-transformed before standardising? Plot a histogram of `d["minimum_nights"]` (log scale): which values stand out?

## A.2 k-means for several k

In [ ]:
rows = []
for k in range(2, 9):
    km_k = KMeans(k, n_init=10, random_state=0).fit(Z)
    rows.append({"k": k, "inertia": km_k.inertia_, "silhouette": silhouette_score(Z, km_k.labels_, sample_size=5000, random_state=0)})
pd.DataFrame(rows).set_index("k").round(3)

## A.3 Hierarchical clustering (Ward) on a sample

In [ ]:
rng = np.random.default_rng(0)
idx = rng.choice(len(Z), size=2000, replace=False)
Z_ward = linkage(Z[idx], method="ward")
fig, ax = plt.subplots(figsize=(10, 3.5))
dendrogram(Z_ward, truncate_mode="lastp", p=25, ax=ax)
ax.set_title("Ward dendrogram of 2,000 listings, top 25 merges")
ward5 = fcluster(Z_ward, t=5, criterion="maxclust")
km = KMeans(5, n_init=10, random_state=0).fit(Z)
print("silhouette on the sample: k-means", round(silhouette_score(Z[idx], km.labels_[idx]), 3),
      "| Ward", round(silhouette_score(Z[idx], ward5), 3),
      "| ARI between them", round(adjusted_rand_score(km.labels_[idx], ward5), 2))

## A.4 DBSCAN on the features and on the map

In [ ]:
dist, _ = NearestNeighbors(n_neighbors=10).fit(Z).kneighbors(Z)
print("10th-neighbour distance quantiles:", np.quantile(dist[:, -1], [0.5, 0.9, 0.99]).round(2))
for eps in (0.5, 0.8, 1.0):
    labels = pd.Series(DBSCAN(eps=eps, min_samples=10).fit_predict(Z))
    print(eps, labels.nunique() - (labels == -1).any(), "clusters,", (labels == -1).sum(), "noise")

In [ ]:
C = d[["east_km", "north_km"]].to_numpy()
d["hot_spot"] = DBSCAN(eps=0.3, min_samples=30).fit_predict(C)
fig, ax = plt.subplots(figsize=(7, 6))
out = d["hot_spot"] == -1
ax.scatter(d.loc[out, "east_km"], d.loc[out, "north_km"], s=2, color="lightgrey", label="outside hot spots")
ax.scatter(d.loc[~out, "east_km"], d.loc[~out, "north_km"], s=2, c=d.loc[~out, "hot_spot"], cmap="tab20")
ax.set(xlabel="km east of Alexanderplatz", ylabel="km north of Alexanderplatz", aspect="equal",
       title="DBSCAN hot spots of Airbnb listings (eps 300 m, 30 listings)")
ax.legend(loc="lower left")
(d[~out].groupby("hot_spot").agg(listings=("id", "size"), area=("neighbourhood", lambda s: s.mode()[0]),
                                  median_price=("price", "median")).nlargest(8, "listings"))

## A.5 Stability and profile of the chosen clustering

In [ ]:
aris = []
for seed in range(10):
    boot = rng.choice(len(Z), size=len(Z), replace=True)
    aris.append(adjusted_rand_score(km.labels_, KMeans(5, n_init=10, random_state=seed).fit(Z[boot]).predict(Z)))
print("bootstrap ARI min / median:", np.round([min(aris), np.median(aris)], 2))
d["cluster"] = km.labels_
d.groupby("cluster").agg(
    listings=("id", "size"), price=("price", "median"), guests=("accommodates", "median"),
    min_nights=("minimum_nights", "median"), open_days=("availability_365", "median"),
    reviews_ltm=("number_of_reviews_ltm", "median"), km_centre=("km_centre", "median"),
    entire_home=("room_type", lambda s: (s == "Entire home/apt").mean()),
    multi_host=("calculated_host_listings_count", lambda s: (s > 1).mean()),
    licence_given=("license", lambda s: s.notna().mean()),
).round(2)

In [ ]:
pd.crosstab(d["district"], d["cluster"], normalize="index").round(2)

**Exercise A.** Write a one-paragraph description of the five clusters for a city official: one sentence per cluster, its size, and the evidence (variables not used in the clustering: room type, district, host with several listings, registration number given). How stable is the segmentation, and how well separated? Which cluster shows a data problem in the price column?

# Part B: what do the listings offer?

## B.1 PCA of the amenity lists

In [ ]:
mlb = MultiLabelBinarizer()
A = pd.DataFrame(mlb.fit_transform(listings["amenities"].apply(json.loads)), columns=mlb.classes_, index=listings.index)
share = A.mean()
A = A.loc[:, (share >= 0.05) & (share <= 0.95)]
pca_a = PCA().fit(A)
print(len(mlb.classes_), "labels;", A.shape[1], "kept; variance of PC1-PC3:", pca_a.explained_variance_ratio_[:3].round(3))
loadings = pd.DataFrame(pca_a.components_[:3].T, index=A.columns, columns=["PC1", "PC2", "PC3"])
pd.concat({pc: loadings[pc].sort_values(ascending=False).head(5).index.to_series(index=range(5))
           for pc in loadings} | {f"{pc} (negative)": loadings[pc].sort_values().head(5).index.to_series(index=range(5))
                                  for pc in loadings}, axis=1)

**Questions.** (a) What does PC1 measure? Correlate its scores with the number of amenities per listing. (b) Look at PC2: `pd.crosstab(A["Heating"], A["Central heating"])`. Is PC2 about the flats, or about how the amenity form was filled in? (c) Name PC3 from its loadings, then check your name against the room types.

## B.2 A t-SNE map of the listings (amenities, 20 components)

In [ ]:
pick = np.random.default_rng(1).choice(len(A), size=3000, replace=False)
S = pca_a.transform(A)[pick, :20]
emb = TSNE(perplexity=30, init="pca", random_state=0).fit_transform(S)
rooms = listings["room_type"].to_numpy()[pick]
fig, ax = plt.subplots(figsize=(7, 6))
for room in ["Entire home/apt", "Private room", "Hotel room", "Shared room"]:
    m = rooms == room
    ax.scatter(*emb[m].T, s=4, label=room)
ax.set(title="t-SNE of 3,000 listings (20 amenity components)", xticks=[], yticks=[])
ax.legend(markerscale=3)

**Question.** Which groups do you see, and which variable explains them? What may you *not* conclude from the distances between the groups (theory page 3)?

# Part C: implausible listings and cluster features

## C.1 Isolation Forest, LOF and the price ratio

Short stays only (minimum below 90 nights): the medium-term rentals have a different kind of price.

In [ ]:
short = listings[listings["price"].notna() & (listings["minimum_nights"] < 90)].reset_index(drop=True)
short["log_price"] = np.log(short["price"])
short["log_min_nights"] = np.log(short["minimum_nights"])
Zs = StandardScaler().fit_transform(short[["log_price", "accommodates", "log_min_nights", "km_centre"]])
short["iso"] = -IsolationForest(random_state=0).fit(Zs).score_samples(Zs)
short["lof"] = -LocalOutlierFactor(n_neighbors=20).fit(Zs).negative_outlier_factor_
X = pd.get_dummies(short[["accommodates", "bedrooms", "east_km", "north_km", "room_type"]], dtype=float)
expected = cross_val_predict(HistGradientBoostingRegressor(random_state=0), X, short["log_price"],
                             cv=KFold(5, shuffle=True, random_state=0))
short["ratio"] = np.exp(short["log_price"] - expected)
show = ["room_type", "property_type", "accommodates", "bedrooms", "price", "minimum_nights", "km_centre", "number_of_reviews"]
print(len(short), "short-stay listings;", ((short["ratio"] > 3) | (short["ratio"] < 1 / 3)).sum(), "off by a factor of 3 or more")
short.nlargest(8, "ratio")[show + ["ratio", "iso", "lof"]].round(2)

In [ ]:
short.nsmallest(8, "ratio")[show + ["ratio", "iso", "lof"]].round(2)

In [ ]:
short.nlargest(8, "iso")[show + ["ratio", "iso", "lof"]].round(2)

**Exercise C.1.** For the top eight of each list decide: probably an error or placeholder, probably genuine but rare, or unclear? What would a person need to check on the listing page? How many listings per week could a city office check, and what threshold for the ratio would that imply?

## C.2 Do location clusters improve the price model?

Gradient boosting for the log price (as in Session 10), folds grouped by host. Three variants: district only, plus distances to 20 k-means location clusters (fitted inside each fold), plus raw coordinates.

In [ ]:
def price_model(location=None, k=20):
    parts = [(OrdinalEncoder(), ["room_type", "district"]),
             ("passthrough", ["accommodates", "bedrooms", "minimum_nights"])]
    if location == "clusters":
        parts.append((KMeans(n_clusters=k, n_init=10, random_state=0), ["east_km", "north_km"]))
    if location == "coordinates":
        parts.append(("passthrough", ["east_km", "north_km"]))
    return make_pipeline(make_column_transformer(*parts),
                         HistGradientBoostingRegressor(categorical_features=[0, 1], random_state=0))


rows = []
for name, location in [("district only", None), ("+ 20 location clusters", "clusters"), ("+ raw coordinates", "coordinates")]:
    mae = -cross_val_score(price_model(location), short, short["log_price"], groups=short["host_id"],
                           cv=GroupKFold(n_splits=5), scoring="neg_mean_absolute_error")
    rows.append({"features": name, "MAE (log)": mae.mean(), "sd across folds": mae.std(),
                 "typical factor off": np.exp(mae.mean())})
pd.DataFrame(rows).set_index("features").round(3)

**Questions.** (1) Is the gain from location clusters larger than the spread across folds? (2) Why do raw coordinates work as well for a tree? (3) Repeat with a linear model (`Ridge` after one-hot encoding) instead of gradient boosting: do the clusters help more there? (4) Why must k-means be fitted inside the pipeline?

## Exercises

1. Add the PC1 amenity score as a seventh clustering feature. How do the clusters change? Which choice of features answers the question "what kinds of offers exist" better?
2. Cluster only the entire homes with at most 30 minimum nights. Which segments appear now?
3. Use HDBSCAN (`sklearn.cluster.HDBSCAN(min_cluster_size=50)`) on the map coordinates. How do its hot spots compare with DBSCAN's?
4. Compute the share of listings with a registration number (`license` not missing) per cluster and per hot spot. Where would a city office look first? (Remember that some values in `license` are placeholders.)

In [ ]:
# Exercise 1: your code here